In [2]:
import os
from dotenv import load_dotenv

# 1. Kiểm tra xem đang ở thư mục nào
print("Thư mục hiện tại:", os.getcwd())

# 2. Nếu đang ở trong thư mục 'notebooks', lùi ra ngoài thư mục gốc
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print("Đã chuyển thư mục ra root:", os.getcwd())

# 3. Load các biến môi trường từ file .env
load_dotenv()
print("Đã load file .env thành công!")

Thư mục hiện tại: /home/huyy-giaa/Develop/VietCulture_Rag/notebooks
Đã chuyển thư mục ra root: /home/huyy-giaa/Develop/VietCulture_Rag
Đã load file .env thành công!


In [3]:
import json
import pandas as pd
import numpy as np

df = pd.read_json("notebooks/eval_results_test3.json")

In [4]:
# Xem 1 chunk text thực tế
df = pd.read_json("notebooks/eval_results_test3.json")
row = df.iloc[0]

print("=== Ground Truth Chunk ID ===")
print(row["ground_truth_chunk"])

print("\n=== Contexts[0] (chunk text đầu tiên) ===")
print(row["contexts"][0])

print("\n=== Contexts[1] ===")
print(row["contexts"][1])

=== Ground Truth Chunk ID ===
am_thuc:banh bao:000015:3:y nghia van hoa cua banh bao trong am thuc viet nam la gi

=== Contexts[0] (chunk text đầu tiên) ===
Topic: banh bao

Normalized Topic:
banh bao

Category:
am_thuc

Cultural Knowledge:
Bánh bao phản ánh sự đa dạng và thích nghi của ẩm thực Việt.

Detailed Explanation:
Bánh bao, dù có nguồn gốc từ bên ngoài, đã được người Việt tiếp nhận và biến tấu thành một phần không thể thiếu của nền ẩm thực. Sự đa dạng về nhân bánh bao thể hiện sự sáng tạo và khéo léo trong việc chế biến. Bánh bao đóng gói thể hiện sự phát triển của công nghiệp thực phẩm, đáp ứng nhu cầu của xã hội hiện đại.

Representative Question:
Ý nghĩa văn hóa của bánh bao trong ẩm thực Việt Nam là gì?

=== Contexts[1] ===
Topic: banh bao

Normalized Topic:
banh bao

Category:
am_thuc

Cultural Knowledge:
Phản ánh nét văn hoá ẩm thực truyền thống.

Detailed Explanation:
Hình ảnh phản ánh nét văn hóa ẩm thực truyền thống của Việt Nam với việc sử dụng xửng hấp bằng tre, một

In [5]:
import pandas as pd
import numpy as np

df = pd.read_json("notebooks/eval_results_test3.json")

# ── Parse category và keyword từ chunk text ────────────
def parse_chunk_key(chunk_text):
    """
    Parse "Category: am_thuc" và "Normalized Topic: banh bao"
    từ chunk text, trả về "am_thuc:banh bao"
    """
    category = ""
    keyword  = ""
    lines    = chunk_text.split("\n")
    
    for i, line in enumerate(lines):
        line = line.strip()
        
        if line == "Category:":
            # Dòng tiếp theo là giá trị
            if i + 1 < len(lines):
                category = lines[i + 1].strip()
        
        if line == "Normalized Topic:":
            # Dòng tiếp theo là giá trị
            if i + 1 < len(lines):
                keyword = lines[i + 1].strip()
    
    return f"{category}:{keyword}"

# ── Extract prefix từ ground_truth_chunk_id ────────────
def get_gt_prefix(ground_truth_chunk_id):
    """
    "am_thuc:banh bao:000015:3:y nghia..."
    → "am_thuc:banh bao"
    """
    parts = ground_truth_chunk_id.split(":")
    if len(parts) >= 2:
        return f"{parts[0]}:{parts[1]}"
    return ""

# ── Verify trước khi tính Hit Rate ─────────────────────
print("=== VERIFY 3 samples đầu ===")
for i in range(min(3, len(df))):
    row       = df.iloc[i]
    gt        = row["ground_truth_chunk"]
    gt_prefix = get_gt_prefix(gt)
    
    print(f"\nSample {i+1}:")
    print(f"  GT prefix      : {gt_prefix}")
    
    for j, ctx in enumerate(row["contexts"][:3]):
        parsed = parse_chunk_key(ctx)
        match  = "✅" if parsed == gt_prefix else "❌"
        print(f"  Context[{j}] key : {parsed} {match}")

=== VERIFY 3 samples đầu ===

Sample 1:
  GT prefix      : am_thuc:banh bao
  Context[0] key : am_thuc:banh bao ✅
  Context[1] key : am_thuc:banh bao ✅
  Context[2] key : am_thuc:banh bao ✅

Sample 2:
  GT prefix      : am_thuc:banh bot loc
  Context[0] key : am_thuc:banh bot loc ✅
  Context[1] key : am_thuc:banh bot loc ✅
  Context[2] key : am_thuc:banh day ❌

Sample 3:
  GT prefix      : am_thuc:banh can phan thiet
  Context[0] key : am_thuc:banh can phan thiet ✅
  Context[1] key : le_hoi:le hoi banh chung ❌
  Context[2] key : am_thuc:cha que hung yen ❌


In [6]:
import pandas as pd
import numpy as np

df = pd.read_json("notebooks/eval_results_test3.json")

# ── Hàm parse ──────────────────────────────────────────
def parse_chunk_key(chunk_text):
    category = ""
    keyword  = ""
    lines    = chunk_text.split("\n")
    for i, line in enumerate(lines):
        line = line.strip()
        if line == "Category:":
            if i + 1 < len(lines):
                category = lines[i + 1].strip()
        if line == "Normalized Topic:":
            if i + 1 < len(lines):
                keyword = lines[i + 1].strip()
    return f"{category}:{keyword}"

def get_gt_prefix(gt):
    parts = gt.split(":")
    return f"{parts[0]}:{parts[1]}" if len(parts) >= 2 else ""

# ── Hit Rate ────────────────────────────────────────────
def hit_rate(df, k):
    hits = 0
    for _, row in df.iterrows():
        gt_prefix = get_gt_prefix(row["ground_truth_chunk"])
        keys = [
            parse_chunk_key(ctx) 
            for ctx in row["contexts"][:k]
        ]
        if gt_prefix in keys:
            hits += 1
    return hits / len(df)

# ── Kết quả tổng ───────────────────────────────────────
print("══ Retrieval Metrics ══════════════════")
print(f"Hit Rate @1 : {hit_rate(df, 1):.3f}")
print(f"Hit Rate @3 : {hit_rate(df, 3):.3f}")
print(f"Hit Rate @5 : {hit_rate(df, 5):.3f}")

# ── Breakdown theo Category ─────────────────────────────
print("\n══ Hit Rate @5 theo Category ══════════")
for cat, group in df.groupby("category"):
    print(f"  {cat:25s} : {hit_rate(group, 5):.3f}  ({len(group)} samples)")

# ── Breakdown theo Difficulty ───────────────────────────
print("\n══ Hit Rate @5 theo Difficulty ════════")
for diff, group in df.groupby("difficulty"):
    print(f"  {diff:10s} : {hit_rate(group, 5):.3f}  ({len(group)} samples)")

══ Retrieval Metrics ══════════════════
Hit Rate @1 : 1.000
Hit Rate @3 : 1.000
Hit Rate @5 : 1.000

══ Hit Rate @5 theo Category ══════════
  am_thuc                   : 1.000  (3 samples)

══ Hit Rate @5 theo Difficulty ════════
  medium     : 1.000  (3 samples)


In [7]:
import json
import pandas as pd
from tqdm import tqdm
import time
from src.agent.graph import create_agent_bundle, invoke_agent 

bundle = create_agent_bundle()

with open("notebooks/output_benchmark.json") as f:
    data = json.load(f)

results = []

# SỬA CHỖ NÀY 1: Thay data[:3] thành data để chạy toàn bộ 240 câu
for sample in tqdm(data, desc="Đang đánh giá toàn bộ"): 
    user_query = sample["user_query"]
    conv_id = f"eval_{sample.get('benchmark_id', '0')}" 
    
    # Lấy tài liệu
    retrieved_docs = bundle.retriever.retrieve(query=user_query)
    retrieved_texts = [chunk.document.page_content for chunk in retrieved_docs]
    
    # Giữ nguyên cách lấy doc_id (hoặc ID tương đương) mà bạn đã cấu hình
    retrieved_ids = [chunk.document.metadata.get("doc_id", "") for chunk in retrieved_docs] 
    
    graph_response = invoke_agent(
        bundle=bundle,
        user_id="evaluator_user", 
        conversation_id=conv_id,
        message=user_query
    )
    
    answer = graph_response.get("answer", "") 
    
    results.append({
        "benchmark_id": sample.get("benchmark_id", ""),
        "category": sample.get("category", ""),
        "difficulty": sample.get("difficulty", "unknown"),
        "question": user_query,                      
        "answer": answer,                            
        "contexts": retrieved_texts,                 
        "ground_truth": sample.get("detailed explanation", ""), 
        "retrieved_ids": retrieved_ids,
        "ground_truth_chunk": sample.get("ground_truth_chunk_id", "")
    })
    
    time.sleep(4) 

# SỬA CHỖ NÀY 2: Lưu thành file mới (ví dụ: eval_results_full.json)
pd.DataFrame(results).to_json(
    "notebooks/eval_results_full.json",
    orient="records",
    force_ascii=False
)
print("Done! Đã hoàn thành đánh giá toàn bộ dataset.")

E0000 00:00:1782487966.758928   19799 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782487966.759230   19799 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1782487966.759236   19799 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782487966.759240   19799 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1782487966.759243   19799 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Đang đánh giá toàn bộ:  28%|██▊       | 67/240 [26:59<1:09:41, 24.17s/it]


KeyboardInterrupt: 

In [8]:
import json

# Mở file benchmark ra xem
with open("notebooks/output_benchmark.json", encoding="utf-8") as f:
    data = json.load(f)

for sample in data[:5]:
    print(sample.get("ground_truth_chunk_id"))

am_thuc:banh bao:000015:3:y nghia van hoa cua banh bao trong am thuc viet nam la gi
am_thuc:banh bot loc:000042:3:y nghia van hoa cua viec su dung la chuoi de goi banh bot loc la gi
am_thuc:banh can phan thiet:000022:3:y nghia van hoa cua viec an banh can la gi
am_thuc:banh chay:000037:3:y nghia van hoa cua coi da trong xa hoi viet nam la gi
am_thuc:banh chung gu:000046:3:y nghia van hoa cua banh chung gu la gi
